In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)


In [2]:

# ============================================================
# 1) Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [3]:

# ============================================================
# 2) Files to test
#    Make sure these files exist in your working folder
# ============================================================
DATASETS = [
    "../fraud_preprocessing/fraud_prepared_numeric.csv",
    "../wrapper/fraud_selected_features_rfecv.csv",
    "../PCA/fraud_pca_95_variance.csv"
]

TARGET_COL = "fraud"


In [4]:

# ============================================================
# 3) Build RNN model
#    For tabular data, we reshape each sample to:
#    (timesteps = number_of_features, features_per_timestep = 1)
# ============================================================
def build_rnn_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.GRU(64, return_sequences=True),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.GRU(32),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="roc_auc"),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc")
        ]
    )
    return model


In [5]:

# ============================================================
# 4) Tune threshold on validation set
#    Fraud is imbalanced, so 0.5 is not always best
# ============================================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.05)
    best_threshold = 0.50
    best_f1 = -1

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = thr

    return best_threshold, best_f1


In [6]:

# ============================================================
# 5) Train and evaluate one dataset
# ============================================================
def run_experiment(csv_path, target_col="fraud", epochs=30, batch_size=256):
    print("=" * 80)
    print(f"DATASET: {csv_path}")

    if not os.path.exists(csv_path):
        print(f"File not found: {csv_path}")
        print("Skipping...\n")
        return None

    # ----------------------------
    # Load data
    # ----------------------------
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in {csv_path}")

    # Just in case there are missing values
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if col != target_col:
                df[col] = df[col].fillna(df[col].median())

    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].astype(int).copy()

    print(f"Shape: {df.shape}")
    print(f"Features: {X.shape[1]}")
    print("Class distribution:")
    print(y.value_counts(normalize=True).rename("proportion"))

    # ----------------------------
    # Split: train / val / test
    # 64% / 16% / 20%
    # ----------------------------
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y,
        test_size=0.20,
        stratify=y,
        random_state=SEED
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.20,
        stratify=y_train_full,
        random_state=SEED
    )

    # ----------------------------
    # Scale features
    # ----------------------------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # ----------------------------
    # Reshape for RNN
    # (samples, timesteps, channels)
    # ----------------------------
    X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_val_rnn = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
    X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    # ----------------------------
    # Class weights
    # ----------------------------
    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    print("Class weights:", class_weight_dict)

    # ----------------------------
    # Model
    # ----------------------------
    model = build_rnn_model(input_shape=(X_train_rnn.shape[1], 1))
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1
        )
    ]

    # ----------------------------
    # Train
    # ----------------------------
    history = model.fit(
        X_train_rnn, y_train,
        validation_data=(X_val_rnn, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

    # ----------------------------
    # Validation threshold tuning
    # ----------------------------
    val_prob = model.predict(X_val_rnn, verbose=0).ravel()
    best_threshold, best_val_f1 = find_best_threshold(y_val, val_prob)

    print(f"Best validation threshold: {best_threshold:.2f}")
    print(f"Best validation F1: {best_val_f1:.4f}")

    # ----------------------------
    # Test evaluation
    # ----------------------------
    test_prob = model.predict(X_test_rnn, verbose=0).ravel()
    test_pred = (test_prob >= best_threshold).astype(int)

    metrics = {
        "dataset": csv_path,
        "n_features": X.shape[1],
        "best_threshold": best_threshold,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, test_prob),
        "pr_auc": average_precision_score(y_test, test_prob)
    }

    cm = confusion_matrix(y_test, test_pred)

    print("\nTest Metrics")
    for k, v in metrics.items():
        if k not in ["dataset", "n_features"]:
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    print("\nConfusion Matrix")
    print(cm)

    # ----------------------------
    # Save outputs
    # ----------------------------
    base_name = os.path.splitext(os.path.basename(csv_path))[0]

    model_dir = os.path.join("..", "model", "RNN")
    csv_dir = os.path.join(model_dir, "csv")

    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(csv_dir, exist_ok=True)

    history_path = os.path.join(csv_dir, f"{base_name}_rnn_history.csv")
    pred_path = os.path.join(csv_dir, f"{base_name}_rnn_test_predictions.csv")
    model_path = os.path.join(model_dir, f"{base_name}_rnn_model.keras")

    pd.DataFrame(history.history).to_csv(history_path, index=False)

    pd.DataFrame({
        "y_true": y_test.reset_index(drop=True),
        "y_prob": test_prob,
        "y_pred": test_pred
    }).to_csv(pred_path, index=False)

    model.save(model_path)

    print("\nSaved:")
    print(f"- {history_path}")
    print(f"- {pred_path}")
    print(f"- {model_path}")

    return metrics


In [7]:

# ============================================================
# 6) Run on all datasets
# ============================================================
all_results = []

for file_name in DATASETS:
    result = run_experiment(file_name, target_col=TARGET_COL, epochs=30, batch_size=256)
    if result is not None:
        all_results.append(result)


DATASET: ../fraud_preprocessing/fraud_prepared_numeric.csv
Shape: (180519, 56)
Features: 55
Class distribution:
fraud
0    0.977498
1    0.022502
Name: proportion, dtype: float64
Class weights: {0: 0.5115113519640138, 1: 22.217692307692307}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 55, 64)         │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 55, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,361 (91.25 KB)

 Trainable params: 23,361 (91.25 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 11, 64)         │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 11, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,361 (91.25 KB)

 Trainable params: 23,361 (91.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
452/452 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.4774 - loss: 0.6938 - pr_auc: 0.0222 - precision: 0.0219 - recall: 0.5096 - roc_auc: 0.4952 - val_accuracy: 0.5774 - val_loss: 0.6883 - val_pr_auc: 0.0229 - val_precision: 0.0229 - val_recall: 0.4262 - val_roc_auc: 0.5085 - learning_rate: 0.0010
Epoch 2/30
452/452 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.4492 - loss: 0.6935 - pr_auc: 0.0227 - precision: 0.0225 - recall: 0.5527 - roc_auc: 0.5020 - val_accuracy: 0.4666 - val_loss: 0.6928 - val_pr_auc: 0.0231 - val_precision: 0.0242 - val_recall: 0.5769 - val_roc_auc: 0.5126 - learning_rate: 0.0010
Epoch 3/30
452/452 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.5139 - loss: 0.6936 - pr_auc: 0.0226 - precision: 0.0228 - recall: 0.4927 - roc_auc: 0.5043 - val_accuracy: 0.7523 - val_loss: 0.6779 - val_pr_auc: 0.0236 - val_precision: 0.0253 - val_recall: 0.2662 - val_roc_auc: 0.5175 - learning_rate: 0.0010
Epoch 4/30
452/452 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accur

In [ ]:

# ============================================================
# 7) Compare results
# ============================================================
if len(all_results) > 0:
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(by="pr_auc", ascending=False)

    print("\n" + "=" * 80)
    print("FINAL COMPARISON")
    print(results_df.to_string(index=False))

    model_dir = os.path.join("..", "model", "RNN")
    csv_dir = os.path.join(model_dir, "csv")

    results_df.to_csv(
    os.path.join(csv_dir, "rnn_results_comparison.csv"),
    index=False
    )
    print("\nSaved: rnn_results_comparison.csv")
else:
    print("No dataset was successfully processed.")


FINAL COMPARISON
                                          dataset  n_features  best_threshold  accuracy  precision   recall       f1  roc_auc   pr_auc
                 ../PCA/fraud_pca_95_variance.csv          11             0.5  0.508946   0.024403 0.534483 0.046674 0.525339 0.024331
../fraud_preprocessing/fraud_prepared_numeric.csv          55             0.1  0.022491   0.022491 1.000000 0.043992 0.509998 0.024133
     ../wrapper/fraud_selected_features_rfecv.csv          17             0.5  0.725238   0.024734 0.291872 0.045603 0.518238 0.023666

Saved: rnn_results_comparison.csv
